# Notebook to test the scOPE code

Andrew Ashford, Pathways + Omics Group, OHSU
5/20/2024

This workflow will use bulk RNA-seq counts and variant call data to train logistic ridge regression classifiers to predict the presence or absence of cancer-associated mutations. I will then apply the models to single-cell data and perform additional experiments to determine clonal structures. This notebook is meant to implement and debug/test the code for this workflow as I write it.

In [1]:
# Import modules
import sys
import pandas as pd
import numpy as np

# Import custom modules
sys.path.append('../')
from scOPE import preprocessing
from scOPE import utilities
from scOPE import models
from scOPE import evaluate


## Training classification models using bulk RNA-seq data

#### Read and preprocess data

In [2]:
# Specify directory and filename of Variant Call File (VCF)
vcf_dir = '/home/groups/precepts/ashforda/scOPE_github_stuff/data/training/BeatAML/'
vcf_filename = 'beataml_wes_wv1to4_mutations_v4_exon_mut_loc_and_variant_class_added.tsv'


In [3]:
# Read VCF file into a Pandas dataframe with this preprocessing function
# NOTE: This function defaults to delimiter='\t', to change this, add a delimiter= flag and specify delimiter
vcf_df = preprocessing.tsv_to_df(vcf_dir + vcf_filename)


In [4]:
# Sanity check the dataframe of the tsv file
print(vcf_df)


      seqnames  pos_start   pos_end ref alt genotyper     wave    seq_id  \
0            1     914477    914477   C   T    mutect  wave1+2  14-00141   
1            1     914477    914477   C   T   varscan  wave1+2  14-00141   
2            1     914941    914941   G   A    mutect  wave1+2  15-00492   
3            1     914941    914941   G   A   varscan  wave1+2  15-00492   
4            1     982281    982281   G   T   varscan  wave1+2  20-00157   
...        ...        ...       ...  ..  ..       ...      ...       ...   
11716        6   35259365  35259365   G   A   varscan  wave3+4  40-00111   
11717        X   15841107  15841108  GA   G   varscan  wave3+4  30-00644   
11718        X   15833799  15833799   G   C   varscan  wave3+4  40-00156   
11719       17    3912933   3912933   T   A    mutect  wave3+4  30-00878   
11720       17    3912933   3912933   T   A    mutect  wave3+4  30-00888   

      original_id  tumor_only  ...  protein_position  amino_acids   codons  \
0        

In [5]:
# Eliminate sex chromosomes
# Filter out rows where 'seqnames' is 'X' or 'Y'
vcf_df = vcf_df[~vcf_df['seqnames'].isin(['X', 'Y'])]


In [6]:
# Display the filtered DataFrame
print(vcf_df)


      seqnames  pos_start   pos_end ref alt genotyper     wave    seq_id  \
0            1     914477    914477   C   T    mutect  wave1+2  14-00141   
1            1     914477    914477   C   T   varscan  wave1+2  14-00141   
2            1     914941    914941   G   A    mutect  wave1+2  15-00492   
3            1     914941    914941   G   A   varscan  wave1+2  15-00492   
4            1     982281    982281   G   T   varscan  wave1+2  20-00157   
...        ...        ...       ...  ..  ..       ...      ...       ...   
11714        6   35259365  35259365   G   A   varscan  wave3+4  30-00907   
11715        6   35259365  35259365   G   A    mutect  wave3+4  40-00111   
11716        6   35259365  35259365   G   A   varscan  wave3+4  40-00111   
11719       17    3912933   3912933   T   A    mutect  wave3+4  30-00878   
11720       17    3912933   3912933   T   A    mutect  wave3+4  30-00888   

      original_id  tumor_only  ...  protein_position  amino_acids   codons  \
0        

In [7]:
# Filter out the rows where the "polyphen" column doesn't contain the word "damaging"
#vcf_df = vcf_df[vcf_df['polyphen'].str.contains('damaging', na=False)]
vcf_df = vcf_df[vcf_df['existing_variation'].str.contains('COSM', na=False)]


In [8]:
# Display the filtered DataFrame
print(vcf_df)


      seqnames  pos_start   pos_end   ref  alt genotyper     wave    seq_id  \
17           1    1747229   1747229     T    C    mutect  wave1+2  20-00127   
30           1    5934587   5934587     C    T   varscan  wave1+2  20-00353   
31           1    5934587   5934587     C    T   varscan  wave1+2  20-00237   
42           1    6680068   6680071  TGAA    T   varscan  wave1+2  20-00156   
43           1    6680068   6680071  TGAA    T   varscan  wave1+2  15-00514   
...        ...        ...       ...   ...  ...       ...      ...       ...   
11673       11   32417910  32417910     G    T   varscan  wave3+4  40-00039   
11675       11   32417911  32417911     A   AC   varscan  wave3+4  40-00168   
11677       11   32417923  32417923     T  TCC   varscan  wave3+4  40-00146   
11679       11   32417941  32417941     C   CA   varscan  wave3+4  40-00144   
11680       11   32417942  32417942     A   AG   varscan  wave3+4  40-00144   

      original_id  tumor_only  ...  protein_positio

In [9]:
# Specify directory and filename of the RNA-seq counts file
counts_dir = '/home/groups/precepts/ashforda/scOPE_github_stuff/data/training/BeatAML/'
#counts_filename = 'beataml_waves1to4_norm_transposed.tsv'
counts_filename = 'beataml_waves1to4_norm.tsv'


In [10]:
# Read RNA-seq counts TSV file into a Pandas dataframe with this preprocessing function
# NOTE: This function defaults to delimiter='\t', to change this, add a delimiter= flag and specify delimiter
counts_df = preprocessing.tsv_to_df(counts_dir + counts_filename, index_col = 0)


In [11]:
# Print DF as a sanity check
print(counts_df)

                 display_label  \
stable_id                        
ENSG00000000003         TSPAN6   
ENSG00000000419           DPM1   
ENSG00000000457          SCYL3   
ENSG00000000460       C1orf112   
ENSG00000000938            FGR   
...                        ...   
ENSG00000273477  RP11-196O16.1   
ENSG00000273483   RP4-671G15.2   
ENSG00000273486  RP11-731C17.2   
ENSG00000273487   RP4-621B10.8   
ENSG00000273488   RP11-114I8.4   

                                                       description  \
stable_id                                                            
ENSG00000000003       tetraspanin 6 [Source:HGNC Symbol;Acc:11858]   
ENSG00000000419  dolichyl-phosphate mannosyltransferase polypep...   
ENSG00000000457  SCY1-like 3 (S. cerevisiae) [Source:HGNC Symbo...   
ENSG00000000460  chromosome 1 open reading frame 112 [Source:HG...   
ENSG00000000938  feline Gardner-Rasheed sarcoma viral oncogene ...   
...                                                            ... 

In [12]:
print(set(counts_df['biotype']))


{'TR_V_gene', 'rRNA', 'IG_J_gene', 'antisense', 'miRNA', 'TR_J_gene', 'polymorphic_pseudogene', 'IG_C_gene', 'lincRNA', 'IG_V_pseudogene', 'misc_RNA', 'IG_D_gene', 'sense_overlapping', 'snoRNA', 'processed_transcript', 'Mt_tRNA', 'IG_J_pseudogene', 'pseudogene', '3prime_overlapping_ncrna', 'Mt_rRNA', 'IG_V_gene', 'snRNA', 'TR_C_gene', 'protein_coding', 'IG_C_pseudogene', 'sense_intronic', 'TR_V_pseudogene'}


In [13]:
# Should probably eliminate some features based on what their biotype is
# For instance, maybe should use miRNA, protein_coding, processed_transcript
#filtered_df = counts_df[counts_df['biotype'].isin(['miRNA', 'protein_coding', 'processed_transcript'])]
filtered_df = counts_df[counts_df['biotype'].isin(['protein_coding'])]

# Should maybe stick with only protein-coding genes and eliminate sex chromosomes. 
# Maybe eliminate the chromosome that the mutation of interest is on.


In [14]:
print(filtered_df)
print(filtered_df.shape)


                  display_label  \
stable_id                         
ENSG00000000003          TSPAN6   
ENSG00000000419            DPM1   
ENSG00000000457           SCYL3   
ENSG00000000460        C1orf112   
ENSG00000000938             FGR   
...                         ...   
ENSG00000272916  RP11-574K11.31   
ENSG00000273045         C2ORF15   
ENSG00000273154   RP4-583P15.15   
ENSG00000273173           SNURF   
ENSG00000273294   RP11-1084J3.4   

                                                       description  \
stable_id                                                            
ENSG00000000003       tetraspanin 6 [Source:HGNC Symbol;Acc:11858]   
ENSG00000000419  dolichyl-phosphate mannosyltransferase polypep...   
ENSG00000000457  SCY1-like 3 (S. cerevisiae) [Source:HGNC Symbo...   
ENSG00000000460  chromosome 1 open reading frame 112 [Source:HG...   
ENSG00000000938  feline Gardner-Rasheed sarcoma viral oncogene ...   
...                                                   

In [15]:
# Filter the counts DF so it only contains the sample RNA counts and their ENSG row labels.
# Filter columns where names contain a dash '-'
counts_only_df = filtered_df.loc[:, filtered_df.columns.str.contains('-')]

print(counts_only_df)

# Transpose the dataframe so that samples are the rows and the columns are genes for data scaling
counts_only_df = counts_only_df.transpose()

print(counts_only_df)


                 13-00098  13-00342  13-00353  13-00466  13-00468  13-00500  \
stable_id                                                                     
ENSG00000000003 -3.785178 -1.436271 -1.439675  0.798084 -2.731208 -1.126240   
ENSG00000000419  7.263234  6.967032  7.355790  7.638786  6.960916  7.209736   
ENSG00000000457  3.548906  3.470017  3.991816  4.281776  3.817372  3.426501   
ENSG00000000460  4.263616  3.081390  3.177304  1.560156  2.795027  3.185943   
ENSG00000000938  4.158187  6.205253  9.501260  8.652853  9.917637  6.407852   
...                   ...       ...       ...       ...       ...       ...   
ENSG00000272916  0.442049  0.366869 -1.030400  2.112345  0.187685  0.269777   
ENSG00000273045  2.312404  0.601633 -0.799403  2.189262 -0.327714 -0.351438   
ENSG00000273154 -1.107497 -1.270364 -1.326671 -1.384004 -0.227067 -1.280294   
ENSG00000273173 -0.265767 -3.303417 -0.967118  1.981454 -0.909110  1.578374   
ENSG00000273294  0.935171 -0.819108 -2.345934 -0.628

In [16]:
# Data is already TPM normalized, so I'm just selecting to scale in the below function
normalized_scaled_bulk_rna = preprocessing.preprocess_bulk_RNA(counts_only_df, normalize=False, scale=True)

print(normalized_scaled_bulk_rna)
#print(normalized_scaled_bulk_rna.loc[['13-00098', '19-00261']])


Scaling data..
stable_id  ENSG00000000003  ENSG00000000419  ENSG00000000457  ENSG00000000460  \
13-00098         -1.386027         1.029861        -0.418889         1.846032   
13-00342         -0.065975         0.060619        -0.568180         0.094765   
13-00353         -0.067888         1.332723         0.419281         0.236844   
13-00466          1.189700         2.258752         0.968007        -2.158686   
13-00468         -0.793711         0.040606         0.089160        -0.329433   
...                    ...              ...              ...              ...   
19-00400         -0.810371        -0.564297        -0.122115        -0.685624   
18-00179          0.411533         0.223729         0.970081         0.641550   
19-00092         -0.528849         0.615857        -0.053878         1.769788   
19-00200          0.228428         0.235660         0.463599         0.086589   
19-00261         -1.030377         0.094259        -0.380782         0.348351   

stable_id  E

In [17]:
# Select the top 2000 most variable genes in bulk RNA-seq data
#normalized_scaled_bulk_rna = utilities.select_top_variable_genes_bulk_rnaseq(normalized_scaled_bulk_rna, 2000)


In [18]:
# Save the above object to reduce the scRNA-seq data to the same variable features
import pickle

# Specify output directory for trained model pickle files
output_dir = '/home/groups/precepts/ashforda/scOPE_github_stuff/trained_models/'

# Save the data dictionary to a pickle file for downstream analysis
with open(output_dir + 'protein_coding_and_miRNA_and_processed_transcripts_bulk_rna.pkl', 'wb') as f:
    pickle.dump(normalized_scaled_bulk_rna, f)



In [19]:
# Generate a list of known AML mutations to generate model-training sets for downstream use
aml_genes = [
    'FLT3', 'NPM1', 'CEBPA', 'RUNX1', 'AML1', 'TP53', 'ASXL1',
    'DNMT3A', 'IDH1', 'IDH2', 'TET2', 'KIT', 'CD117', 'KRAS', 
    'NRAS', 'PTPN11', 'NF1', 'SRSF2', 'U2AF1', 'U2AF35', 'SF3B1', 
    'SF3A1', 'ZRSR2', 'RAD21', 'STAG1', 'STAG2', 'SMC3', 'NOTCH2', 
    'BCOR', 'BCORL1', 'WT1', 'PHF6', 'SH2B3', 'ATM', 'SETD2', 'GATA2'
]


In [20]:
# Fetching Ensembl IDs
# NOTE: The following function defaults to the GRCh37 ENSEMBL human genome version, to change this, change the 
# server variable within the function.
gene_ids = utilities.fetch_ensembl_ids(aml_genes)
print(gene_ids)

# NOTE: Can also perform the following to fetch gene names from ENSEMBL IDs
#ensembl_ids = ['ENSG00000139618', 'ENSG00000240563', 'ENSG00000132854']
#gene_names = utilities.fetch_gene_names_from_ids(ensembl_ids)
#print(gene_names)

Fetching ENSEMBL ID for gene: 0 out of: 36
{'FLT3': 'ENSG00000122025', 'NPM1': 'ENSG00000181163', 'CEBPA': 'ENSG00000245848', 'RUNX1': 'ENSG00000159216', 'AML1': 'Error fetching ID', 'TP53': 'ENSG00000141510', 'ASXL1': 'ENSG00000171456', 'DNMT3A': 'ENSG00000119772', 'IDH1': 'ENSG00000138413', 'IDH2': 'ENSG00000182054', 'TET2': 'ENSG00000168769', 'KIT': 'ENSG00000157404', 'CD117': 'Error fetching ID', 'KRAS': 'ENSG00000133703', 'NRAS': 'ENSG00000213281', 'PTPN11': 'ENSG00000179295', 'NF1': 'ENSG00000196712', 'SRSF2': 'ENSG00000161547', 'U2AF1': 'ENSG00000160201', 'U2AF35': 'Error fetching ID', 'SF3B1': 'ENSG00000115524', 'SF3A1': 'ENSG00000099995', 'ZRSR2': 'ENSG00000169249', 'RAD21': 'ENSG00000164754', 'STAG1': 'ENSG00000118007', 'STAG2': 'ENSG00000101972', 'SMC3': 'ENSG00000108055', 'NOTCH2': 'ENSG00000134250', 'BCOR': 'ENSG00000183337', 'BCORL1': 'ENSG00000085185', 'WT1': 'ENSG00000184937', 'PHF6': 'ENSG00000156531', 'SH2B3': 'ENSG00000111252', 'ATM': 'ENSG00000149311', 'SETD2': 'ENS

#### Iterate through all the gene IDs and pull needed information from the dataframes

This step will create the variables needed to train the model(s)

In [21]:
# Specify the model training dictionary to populate for later model training
model_training_df_dict = {}

# Select mutect or varscan depending on which genotyper you want to use
#genotyper = 'mutect'
#genotyper = 'varscan'

# Number of positive samples necessary
min_samps = 8

# Random seed to start
s = 0

# Iterate through the dictionary
for gene_name, gene_id in gene_ids.items():
    
    # Filter the dataframe to just the genes of interest
    filtered_df = vcf_df[vcf_df['gene'] == gene_id]
    
    # This is a way to drop duplicate samples in the dataset
    # Several of the variants were called for the same sample for both genotypers
    #filtered_df = filtered_df[filtered_df['genotyper'] == genotyper]
    
    # Extract unique original_ids from the given DataFrame
    # Gets all the sample IDs to pull from the RNA data
    unique_sample_ids = filtered_df['original_id'].unique()
        
    # Make sure we have the indices in the RNA-seq data
    unique_sample_ids = [id for id in unique_sample_ids if id in normalized_scaled_bulk_rna.index]
    
    # Check to make sure the resulting dataframe shape has 10 or more samples
    # This is to make sure we have adequate positive samples for training downstream
    if len(unique_sample_ids) >= min_samps:
        if gene_name not in model_training_df_dict.keys():
            model_training_df_dict[gene_name] = {}
            model_training_df_dict[gene_name]['gene_id'] = gene_id
        
        # Add the count of positive samples to the dictionary
        model_training_df_dict[gene_name]['positive_sample_count'] = len(unique_sample_ids)
 
        # Use these original_ids to filter the rows in the RNA-seq data
        # NOTE: This step takes a bit depending on how large your RNA-seq data is
        filtered_rna_seq_data = normalized_scaled_bulk_rna.loc[unique_sample_ids]
        
        # Add the sequencing data from the mutated samples to their own key
        model_training_df_dict[gene_name]['mut_sequencing_data'] = filtered_rna_seq_data
        
        # Randomly sample the same number of negative samples from the remaining RNA-seq data
        # Number of samples to draw, here I'm using the number of positive samples times 4
        num_samples = len(unique_sample_ids)

        # Update the random seed after each iteration
        # Can choose how to increment this number for a more "randomized", yet reproducible effect
        s += 1
        
        # Random seed for reproducibility
        random_seed = s

        # Get the sampled RNA-seq data
        sampled_rna_seq_data = utilities.sample_rna_seq_data(normalized_scaled_bulk_rna, unique_sample_ids, num_samples, random_seed)

        model_training_df_dict[gene_name]['non-mut_sequencing_data'] = sampled_rna_seq_data
        
        
        print(model_training_df_dict.keys())
        print(model_training_df_dict[gene_name].keys())
        
#print(model_training_df_dict)


dict_keys(['FLT3'])
dict_keys(['gene_id', 'positive_sample_count', 'mut_sequencing_data', 'non-mut_sequencing_data'])
dict_keys(['FLT3', 'NPM1'])
dict_keys(['gene_id', 'positive_sample_count', 'mut_sequencing_data', 'non-mut_sequencing_data'])
dict_keys(['FLT3', 'NPM1', 'CEBPA'])
dict_keys(['gene_id', 'positive_sample_count', 'mut_sequencing_data', 'non-mut_sequencing_data'])
dict_keys(['FLT3', 'NPM1', 'CEBPA', 'RUNX1'])
dict_keys(['gene_id', 'positive_sample_count', 'mut_sequencing_data', 'non-mut_sequencing_data'])
dict_keys(['FLT3', 'NPM1', 'CEBPA', 'RUNX1', 'TP53'])
dict_keys(['gene_id', 'positive_sample_count', 'mut_sequencing_data', 'non-mut_sequencing_data'])
dict_keys(['FLT3', 'NPM1', 'CEBPA', 'RUNX1', 'TP53', 'ASXL1'])
dict_keys(['gene_id', 'positive_sample_count', 'mut_sequencing_data', 'non-mut_sequencing_data'])
dict_keys(['FLT3', 'NPM1', 'CEBPA', 'RUNX1', 'TP53', 'ASXL1', 'DNMT3A'])
dict_keys(['gene_id', 'positive_sample_count', 'mut_sequencing_data', 'non-mut_sequencing_d

In [22]:
# Check how many positive samples there are for each gene classification
for key, value in model_training_df_dict.items():
    print(key)
    
    if 'positive_sample_count' in model_training_df_dict[key].keys():
        print(model_training_df_dict[key]['positive_sample_count'])


FLT3
63
NPM1
129
CEBPA
14
RUNX1
44
TP53
55
ASXL1
45
DNMT3A
94
IDH1
49
IDH2
72
TET2
23
KIT
13
KRAS
33
NRAS
96
PTPN11
27
NF1
14
SRSF2
68
U2AF1
32
SF3B1
34
WT1
24
GATA2
17


#### Train the classification models using the data in the gene dictionary

In [23]:
# Specify output directory for trained model pickle files
output_dir = '/home/groups/precepts/ashforda/scOPE_github_stuff/trained_models/'


In [24]:
import pickle

# Save the data dictionary to a pickle file for downstream analysis
with open(output_dir + 'model_training_dictionary.pkl', 'wb') as f:
    pickle.dump(model_training_df_dict, f)


In [25]:
models.logistic_ridge_regression(model_training_df_dict, output_dir, test_size=0.10)


Trained and saved model for gene: FLT3 as FLT3_logistic_ridge_model.pkl
Training set size: 113, Test set size: 13
Evaluation for gene: FLT3
Accuracy: 0.6923076923076923
Precision: 0.625
Recall: 0.8333333333333334
F1 Score: 0.7142857142857143
Confusion Matrix:
[[4 3]
 [1 5]]
Trained and saved model for gene: NPM1 as NPM1_logistic_ridge_model.pkl
Training set size: 232, Test set size: 26
Evaluation for gene: NPM1
Accuracy: 0.9615384615384616
Precision: 0.9444444444444444
Recall: 1.0
F1 Score: 0.9714285714285714
Confusion Matrix:
[[ 8  1]
 [ 0 17]]
Trained and saved model for gene: CEBPA as CEBPA_logistic_ridge_model.pkl
Training set size: 25, Test set size: 3
Evaluation for gene: CEBPA
Accuracy: 0.6666666666666666
Precision: 1.0
Recall: 0.5
F1 Score: 0.6666666666666666
Confusion Matrix:
[[1 0]
 [1 1]]
Trained and saved model for gene: RUNX1 as RUNX1_logistic_ridge_model.pkl
Training set size: 79, Test set size: 9
Evaluation for gene: RUNX1
Accuracy: 0.7777777777777778
Precision: 1.0
Reca

/home/groups/precepts/ashforda/anaconda3/envs/scOPE_working_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Trained and saved model for gene: KIT as KIT_logistic_ridge_model.pkl
Training set size: 23, Test set size: 3
Evaluation for gene: KIT
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
[[1 0]
 [0 2]]
Trained and saved model for gene: KRAS as KRAS_logistic_ridge_model.pkl
Training set size: 59, Test set size: 7
Evaluation for gene: KRAS
Accuracy: 0.5714285714285714
Precision: 0.5
Recall: 0.6666666666666666
F1 Score: 0.5714285714285714
Confusion Matrix:
[[2 2]
 [1 2]]
Trained and saved model for gene: NRAS as NRAS_logistic_ridge_model.pkl
Training set size: 172, Test set size: 20
Evaluation for gene: NRAS
Accuracy: 0.65
Precision: 0.7272727272727273
Recall: 0.6666666666666666
F1 Score: 0.6956521739130435
Confusion Matrix:
[[5 3]
 [4 8]]
Trained and saved model for gene: PTPN11 as PTPN11_logistic_ridge_model.pkl
Training set size: 48, Test set size: 6
Evaluation for gene: PTPN11
Accuracy: 0.8333333333333334
Precision: 0.75
Recall: 1.0
F1 Score: 0.8571428571428571
Co

/home/groups/precepts/ashforda/anaconda3/envs/scOPE_working_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Trained and saved model for gene: GATA2 as GATA2_logistic_ridge_model.pkl
Training set size: 30, Test set size: 4
Evaluation for gene: GATA2
Accuracy: 0.75
Precision: 0.5
Recall: 1.0
F1 Score: 0.6666666666666666
Confusion Matrix:
[[2 1]
 [0 1]]


In [ ]:
models.random_forest_classification(model_training_df_dict, output_dir, test_size=0.10)


Trained and saved model for gene: FLT3 as FLT3_random_forest_model.pkl
Training set size: 113, Test set size: 13
Evaluation for gene: FLT3
Accuracy: 0.7692307692307693
Precision: 0.7142857142857143
Recall: 0.8333333333333334
F1 Score: 0.7692307692307693
Confusion Matrix:
[[5 2]
 [1 5]]
Trained and saved model for gene: NPM1 as NPM1_random_forest_model.pkl
Training set size: 232, Test set size: 26
Evaluation for gene: NPM1
Accuracy: 0.9230769230769231
Precision: 0.8947368421052632
Recall: 1.0
F1 Score: 0.9444444444444444
Confusion Matrix:
[[ 7  2]
 [ 0 17]]
Trained and saved model for gene: CEBPA as CEBPA_random_forest_model.pkl
Training set size: 25, Test set size: 3
Evaluation for gene: CEBPA
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
[[1 0]
 [0 2]]
Trained and saved model for gene: RUNX1 as RUNX1_random_forest_model.pkl
Training set size: 79, Test set size: 9
Evaluation for gene: RUNX1
Accuracy: 0.8888888888888888
Precision: 1.0
Recall: 0.857142857142857

/home/groups/precepts/ashforda/anaconda3/envs/scOPE_working_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Trained and saved model for gene: KIT as KIT_random_forest_model.pkl
Training set size: 23, Test set size: 3
Evaluation for gene: KIT
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Confusion Matrix:
[[1 0]
 [0 2]]
Trained and saved model for gene: KRAS as KRAS_random_forest_model.pkl
Training set size: 59, Test set size: 7
Evaluation for gene: KRAS
Accuracy: 0.42857142857142855
Precision: 0.4
Recall: 0.6666666666666666
F1 Score: 0.5
Confusion Matrix:
[[1 3]
 [1 2]]
Trained and saved model for gene: NRAS as NRAS_random_forest_model.pkl
Training set size: 172, Test set size: 20
Evaluation for gene: NRAS
Accuracy: 0.65
Precision: 0.7777777777777778
Recall: 0.5833333333333334
F1 Score: 0.6666666666666666
Confusion Matrix:
[[6 2]
 [5 7]]
